In [1]:
import glob

from access_moppy import ACCESS_ESM_CMORiser


Loaded Configuration:
Creator Name: Romain Beucher
Organisation: ACCESS-NRI
Creator Email: romain.beucher@anu.edu.au
Creator URL: 0000-0003-3891-5444


In [2]:
import dask.distributed as dask

client = dask.Client(threads_per_worker=1)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 14
Total threads: 14,Total memory: 63.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39055,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40083,Total threads: 1
Dashboard: /proxy/41393/status,Memory: 4.50 GiB
Nanny: tcp://127.0.0.1:36177,


In [3]:
parent_experiment_config = {
    "parent_experiment_id": "piControl",
    "parent_activity_id": "CMIP",
    "parent_source_id": "ACCESS-ESM1-5",
    "parent_variant_label": "r1i1p1f1",
    "parent_time_units": "days since 0001-01-01 00:00:00",
    "parent_mip_era": "CMIP6",
    "branch_time_in_child": 0.0,
    "branch_time_in_parent": 54786.0,
    "branch_method": "standard",
}

In [4]:
ROOT_FOLDER = "/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/"

TARGET_FOLDERS = "output40[0-9]/ocean/"

In [5]:
from pathlib import Path
from typing import List

from access_moppy.utilities import load_model_mappings


def get_monthly_ocean_files(
    compound_name: str,
    root_folder: str = ROOT_FOLDER,
    target_folders: str = TARGET_FOLDERS,
    model_id: str = "ACCESS-ESM1.6",
) -> List[str]:
    """
    Find ocean data files for a given CMOR variable.

    Args:
        compound_name: CMOR compound name (e.g., 'Omon.evs')
        root_folder: Root directory to search for files
        target_folders: Target folder pattern relative to root_folder
        model_id: Model identifier for loading mappings

    Returns:
        List of absolute paths to matching ocean files

    Raises:
        ValueError: If compound_name format is invalid
        FileNotFoundError: If root directory doesn't exist
    """
    # Validate inputs
    if not compound_name or "." not in compound_name:
        raise ValueError(
            f"Invalid compound_name format: {compound_name}. Expected 'table.variable' format."
        )

    # Extract variable name from compound name
    try:
        table_name, variable_name = compound_name.split(".", 1)
    except ValueError:
        raise ValueError(
            f"Invalid compound_name format: {compound_name}. Expected 'table.variable' format."
        )

    # Check if root folder exists
    root_path = Path(root_folder)
    if not root_path.exists():
        raise FileNotFoundError(f"Root folder does not exist: {root_folder}")

    # Load variable mappings
    try:
        mapping = load_model_mappings(compound_name, model_id)
    except Exception as e:
        print(f"Warning: Could not load mapping for {compound_name}: {e}")
        return []

    if not mapping:
        print(f"No mapping found for {compound_name}")
        return []

    # Get model variables from mapping
    model_variables = mapping[variable_name].get("model_variables", [])
    if not model_variables:
        print(f"No model variables found in mapping for {compound_name}")
        return []

    # Search for files
    files_found = []
    search_pattern_base = str(root_path / target_folders)

    for model_variable in model_variables:
        # Ocean files typically have pattern: *-{model_variable}-1monthly-mean*.nc
        filename_pattern = f"*-{model_variable}-1monthly-mean*.nc"
        search_pattern = search_pattern_base + "/" + filename_pattern

        try:
            matching_files = glob.glob(search_pattern)
            files_found.extend(matching_files)

            if not matching_files:
                print(
                    f"No files found for model variable '{model_variable}' with pattern: {search_pattern}"
                )
            else:
                print(
                    f"Found {len(matching_files)} files for model variable '{model_variable}'"
                )

        except Exception as e:
            print(f"Error searching for files with pattern '{search_pattern}': {e}")

    # Remove duplicates and sort
    files_found = sorted(list(set(files_found)))

    if not files_found:
        print(f"No ocean files found for {compound_name} in {search_pattern_base}")
    else:
        print(f"Total files found for {compound_name}: {len(files_found)}")

    return files_found


files = get_monthly_ocean_files("Omon.so")

Found 10 files for model variable 'salt'
Total files found for Omon.so: 10


In [6]:
files

['/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output400/ocean/ocean-3d-salt-1monthly-mean-ym_3076_01.nc',
 '/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output401/ocean/ocean-3d-salt-1monthly-mean-ym_3077_01.nc',
 '/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output402/ocean/ocean-3d-salt-1monthly-mean-ym_3078_01.nc',
 '/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output403/ocean/ocean-3d-salt-1monthly-mean-ym_3079_01.nc',
 '/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output404/ocean/ocean-3d-salt-1monthly-mean-ym_3080_01.nc',
 '/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output405/ocean/ocean-3d-salt-1monthly-mean-ym_3081_01.nc',
 '/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output406/ocean/ocean-3d-salt-1monthly-mean-ym_3082_01.nc',
 '/g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output407/ocean/ocean-3d-salt-1monthly-mean-ym_3083_

In [7]:
cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,
    compound_name="Omon.so",
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,  # <-- This is optional, can be skipped if not needed
)

In [8]:
cmoriser.run()

🗓️  Monthly CMIP6 table detected (Omon.so) - using calendar-aware validation
📂 Opening 10 files with xarray multi-file dataset...


/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:983: UserWarning: Time bounds variable 'time_bnds' missing time units information
  warnings.warn(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:799: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_index = pd.to_datetime(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:861: UserWarning: Could not detect frequency from time coordinate: Out of bounds nanosecond timestamp: 3076-01-16 12:00:00, at position 0
  warnings.warn(f"Could not detect frequency from time coordinate: {e}")
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:308: UserWarning: Multi-file concatenation failed (Could not detect frequency from concatenated time coordinate), falling back to individual file analysis
  warnings.warn(
/g/data/tm

📊 Detecting frequency from time coordinate differences (fallback method)
📁 Analyzing 10 files individually...
📊 Detecting frequency from time coordinate differences (fallback method)
📊 Detecting frequency from time coordinate differences (fallback method)


/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:983: UserWarning: Time bounds variable 'time_bnds' missing time units information
  warnings.warn(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:799: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_index = pd.to_datetime(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:861: UserWarning: Could not detect frequency from time coordinate: Out of bounds nanosecond timestamp: 3078-01-16 12:00:00, at position 0
  warnings.warn(f"Could not detect frequency from time coordinate: {e}")
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:339: UserWarning: Could not detect frequency for file: /g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output402/ocean/ocean-3d-salt-1monthly-mean-ym_3078_01.nc
  war

📊 Detecting frequency from time coordinate differences (fallback method)
📊 Detecting frequency from time coordinate differences (fallback method)
📊 Detecting frequency from time coordinate differences (fallback method)


...

📊 Detecting frequency from time coordinate differences (fallback method)
📊 Detecting frequency from time coordinate differences (fallback method)


/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:983: UserWarning: Time bounds variable 'time_bnds' missing time units information
  warnings.warn(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:799: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_index = pd.to_datetime(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:861: UserWarning: Could not detect frequency from time coordinate: Out of bounds nanosecond timestamp: 3081-01-16 12:00:00, at position 0
  warnings.warn(f"Could not detect frequency from time coordinate: {e}")
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:418: UserWarning: Could not detect frequency for file: /g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output405/ocean/ocean-3d-salt-1monthly-mean-ym_3081_01.nc
  war

📊 Detecting frequency from time coordinate differences (fallback method)
📊 Detecting frequency from time coordinate differences (fallback method)
📊 Detecting frequency from time coordinate differences (fallback method)


/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:983: UserWarning: Time bounds variable 'time_bnds' missing time units information
  warnings.warn(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:799: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_index = pd.to_datetime(
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:861: UserWarning: Could not detect frequency from time coordinate: Out of bounds nanosecond timestamp: 3084-01-16 12:00:00, at position 0
  warnings.warn(f"Could not detect frequency from time coordinate: {e}")
/g/data/tm70/rb5533/code-dev/ACCESS-MOPPy/src/access_moppy/utilities.py:418: UserWarning: Could not detect frequency for file: /g/data/p73/archive/CMIP7/ACCESS-ESM1-6/spinup/Dec25-PI-control/output408/ocean/ocean-3d-salt-1monthly-mean-ym_3084_01.nc
  war

📊 Detecting frequency from time coordinate differences (fallback method)
📊 Detecting frequency from time coordinate differences (fallback method)
🔧 Applying intelligent dataset rechunking...
🔧 Applying dataset rechunking with rules:
  - Time coordinates: single chunk
  - Time bounds: single chunk
  - Data variables: at least 4.0MB chunks
  salt: data variable → time:1, st_ocean:50, yt_ocean:300, xt_ocean:360
  time_bnds: time bounds → single chunk
  xt_ocean: data variable → xt_ocean:360
  yt_ocean: data variable → yt_ocean:300
  st_ocean: data variable → st_ocean:50
  time: coordinate → single chunk
  nv: data variable → nv:2
✅ Dataset rechunking completed
✅ Dataset rechunking completed
🔧 Normalizing missing values to NaN for consistent processing...
✅ Missing values normalized to NaN - XArray will handle propagation correctly
🔧 Applying final CMIP6 missing value standardization for so...
✅ Final CMIP6 missing value applied: 1e+20


In [9]:
ds = cmoriser.to_dataset()

In [10]:
ds

<xarray.Dataset> Size: 3GB
Dimensions:             (time: 120, bnds: 2, lev: 50, j: 300, i: 360,
                         vertices: 4)
Coordinates:
  * time                (time) float64 960B 1.123e+06 1.123e+06 ... 1.127e+06
  * bnds                (bnds) float64 16B 1.0 2.0
  * lev                 (lev) float64 400B 5.0 15.0 25.0 ... 5.499e+03 5.831e+03
  * j                   (j) int64 2kB 0 1 2 3 4 5 6 ... 294 295 296 297 298 299
  * i                   (i) int64 3kB 0 1 2 3 4 5 6 ... 354 355 356 357 358 359
  * vertices            (vertices) int64 32B 0 1 2 3
Data variables:
    time_bnds           (time, bnds) float64 2kB dask.array<chunksize=(120, 2), meta=np.ndarray>
    so                  (time, lev, j, i) float32 3GB dask.array<chunksize=(1, 50, 300, 360), meta=np.ndarray>
    latitude            (j, i) float64 864kB -77.88 -77.88 ... 65.63 65.21
    longitude           (j, i) float64 864kB 80.5 81.5 82.5 ... 79.97 79.99
    vertices_latitude   (j, i, vertices) float64 3MB -78.0 -78.0 ... 65.0 65.42
    vertices_longitude  (j, i, vertices) float64 3MB 80.0 81.0 ... 80.0 80.0
Attributes: (12/44)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    creation_date:          2026-02-11T06:05:14Z
    data_specs_version:     01.00.33
    experiment:             all-forcing simulation of the recent past
    experiment_id:          historical
    ...                     ...
    branch_method:          standard
    external_variables:     areacello deptho volcello
    creator_name:           Romain Beucher
    creator_organisation:   ACCESS-NRI
    creator_email:          romain.beucher@anu.edu.au
    creator_url:            0000-0003-3891-5444

In [11]:
cmoriser.write()

📦 Dataset size: 3.63 GB
   Using chunked writing with DatasetChunker
  Writing time_bnds (120 timesteps/chunk)...
    ✓ time_bnds: 120 timesteps written
  Writing so (1 timesteps/chunk)...
    ✓ so: 120 timesteps written
CMORised output written to so_Omon_ACCESS-ESM1-5_historical_r1i1p1f1_gn_307601-308512.nc
📁 Optimized layout: metadata → data chunks
🗜️ HDF5 compression: shuffle + zlib(level 4) + fletcher32 for data variables


# Test areacello

In [11]:
files = [
    ROOT_FOLDER + "/output401/ocean/ocean-2d-area_t.nc",
    ROOT_FOLDER + "/output401/ocean/ocean-2d-ht.nc",
]

In [12]:
cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,
    compound_name="Ofx.areacello",
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,  # <-- This is optional, can be skipped if not needed
)

In [13]:
cmoriser.run()

✓ Skipping frequency validation for time-independent variable
🔧 Applying intelligent dataset rechunking...
🔧 Applying dataset rechunking with rules:
  - Time coordinates: single chunk
  - Time bounds: single chunk
  - Data variables: at least 4.0MB chunks
  area_t: data variable → time:2, yt_ocean:300, xt_ocean:360
  ht: data variable → time:2, yt_ocean:300, xt_ocean:360
  xt_ocean: data variable → xt_ocean:360
  yt_ocean: data variable → yt_ocean:300
✅ Dataset rechunking completed
✅ Dataset rechunking completed
🔧 Normalizing missing values to NaN for consistent processing...
✅ Missing values normalized to NaN - XArray will handle propagation correctly
🔧 Applying final CMIP6 missing value standardization for areacello...
✅ Final CMIP6 missing value applied: 1e+20


In [14]:
ds = cmoriser.to_dataset()

In [15]:
ds

<xarray.Dataset> Size: 9MB
Dimensions:             (j: 300, i: 360, vertices: 4)
Coordinates:
  * j                   (j) int64 2kB 0 1 2 3 4 5 6 ... 294 295 296 297 298 299
  * i                   (i) int64 3kB 0 1 2 3 4 5 6 ... 354 355 356 357 358 359
  * vertices            (vertices) int64 32B 0 1 2 3
Data variables:
    areacello           (j, i) float32 432kB dask.array<chunksize=(300, 360), meta=np.ndarray>
    latitude            (j, i) float64 864kB -77.88 -77.88 ... 65.63 65.21
    longitude           (j, i) float64 864kB 80.5 81.5 82.5 ... 79.97 79.99
    vertices_latitude   (j, i, vertices) float64 3MB -78.0 -78.0 ... 65.0 65.42
    vertices_longitude  (j, i, vertices) float64 3MB 80.0 81.0 ... 80.0 80.0
Attributes: (12/43)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    creation_date:          2026-02-11T06:05:17Z
    data_specs_version:     01.00.33
    experiment:             all-forcing simulation of the recent past
    experiment_id:          historical
    ...                     ...
    branch_time_in_parent:  54786.0
    branch_method:          standard
    creator_name:           Romain Beucher
    creator_organisation:   ACCESS-NRI
    creator_email:          romain.beucher@anu.edu.au
    creator_url:            0000-0003-3891-5444

In [16]:
cmoriser.write()

📦 Dataset size: 0.01 GB
   Using chunked writing with DatasetChunker
CMORised output written to areacello_Ofx_ACCESS-ESM1-5_historical_r1i1p1f1_gn_fx.nc
📁 Optimized layout: metadata → data chunks
🗜️ HDF5 compression: shuffle + zlib(level 4) + fletcher32 for data variables


## Test thkcello

In [17]:
files = [ROOT_FOLDER + "/output401/ocean/ocean-3d-dzt-1monthly-mean-ym_3077_01.nc"]

In [18]:
cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,
    compound_name="Ofx.thkcello",
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,  # <-- This is optional, can be skipped if not needed
)

In [19]:
cmoriser.run()

✓ Skipping frequency validation for time-independent variable
🔧 Applying intelligent dataset rechunking...
🔧 Applying dataset rechunking with rules:
  - Time coordinates: single chunk
  - Time bounds: single chunk
  - Data variables: at least 4.0MB chunks
  dzt: data variable → time:1, st_ocean:50, yt_ocean:300, xt_ocean:360
  xt_ocean: data variable → xt_ocean:360
  yt_ocean: data variable → yt_ocean:300
  st_ocean: data variable → st_ocean:50
  time: coordinate → single chunk
✅ Dataset rechunking completed
✅ Dataset rechunking completed
🔧 Normalizing missing values to NaN for consistent processing...
✅ Missing values normalized to NaN - XArray will handle propagation correctly
🔧 Applying final CMIP6 missing value standardization for thkcello...
✅ Final CMIP6 missing value applied: 1e+20


In [20]:
ds = cmoriser.to_dataset()

In [21]:
ds

<xarray.Dataset> Size: 30MB
Dimensions:             (time: 12, lev: 50, j: 300, i: 360, vertices: 4)
Coordinates:
  * time                (time) float64 96B 1.124e+06 1.124e+06 ... 1.124e+06
  * lev                 (lev) float64 400B 5.0 15.0 25.0 ... 5.499e+03 5.831e+03
  * j                   (j) int64 2kB 0 1 2 3 4 5 6 ... 294 295 296 297 298 299
  * i                   (i) int64 3kB 0 1 2 3 4 5 6 ... 354 355 356 357 358 359
  * vertices            (vertices) int64 32B 0 1 2 3
Data variables:
    thkcello            (lev, j, i) float32 22MB dask.array<chunksize=(50, 300, 360), meta=np.ndarray>
    latitude            (j, i) float64 864kB -77.88 -77.88 ... 65.63 65.21
    longitude           (j, i) float64 864kB 80.5 81.5 82.5 ... 79.97 79.99
    vertices_latitude   (j, i, vertices) float64 3MB -78.0 -78.0 ... 65.0 65.42
    vertices_longitude  (j, i, vertices) float64 3MB 80.0 81.0 ... 80.0 80.0
Attributes: (12/44)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    creation_date:          2026-02-11T06:05:27Z
    data_specs_version:     01.00.33
    experiment:             all-forcing simulation of the recent past
    experiment_id:          historical
    ...                     ...
    branch_method:          standard
    external_variables:     areacello volcello
    creator_name:           Romain Beucher
    creator_organisation:   ACCESS-NRI
    creator_email:          romain.beucher@anu.edu.au
    creator_url:            0000-0003-3891-5444

In [22]:
cmoriser.write()

📦 Dataset size: 0.04 GB
   Using chunked writing with DatasetChunker
CMORised output written to thkcello_Ofx_ACCESS-ESM1-5_historical_r1i1p1f1_gn_fx.nc
📁 Optimized layout: metadata → data chunks
🗜️ HDF5 compression: shuffle + zlib(level 4) + fletcher32 for data variables


## Test deptho

In [25]:
files = [ROOT_FOLDER + "/output401/ocean/ocean-2d-ht.nc"]

In [26]:
cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,
    compound_name="Ofx.deptho",
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,  # <-- This is optional, can be skipped if not needed
)

In [27]:
cmoriser.run()

✓ Skipping frequency validation for time-independent variable
🔧 Applying intelligent dataset rechunking...
🔧 Applying dataset rechunking with rules:
  - Time coordinates: single chunk
  - Time bounds: single chunk
  - Data variables: at least 4.0MB chunks
  ht: data variable → time:1, yt_ocean:300, xt_ocean:360
  xt_ocean: data variable → xt_ocean:360
  yt_ocean: data variable → yt_ocean:300
✅ Dataset rechunking completed
✅ Dataset rechunking completed
🔧 Normalizing missing values to NaN for consistent processing...
✅ Missing values normalized to NaN - XArray will handle propagation correctly
🔧 Applying final CMIP6 missing value standardization for deptho...
✅ Final CMIP6 missing value applied: 1e+20


In [28]:
ds = cmoriser.to_dataset()

In [29]:
ds

<xarray.Dataset> Size: 9MB
Dimensions:             (j: 300, i: 360, vertices: 4)
Coordinates:
  * j                   (j) int64 2kB 0 1 2 3 4 5 6 ... 294 295 296 297 298 299
  * i                   (i) int64 3kB 0 1 2 3 4 5 6 ... 354 355 356 357 358 359
  * vertices            (vertices) int64 32B 0 1 2 3
Data variables:
    deptho              (j, i) float32 432kB dask.array<chunksize=(300, 360), meta=np.ndarray>
    latitude            (j, i) float64 864kB -77.88 -77.88 ... 65.63 65.21
    longitude           (j, i) float64 864kB 80.5 81.5 82.5 ... 79.97 79.99
    vertices_latitude   (j, i, vertices) float64 3MB -78.0 -78.0 ... 65.0 65.42
    vertices_longitude  (j, i, vertices) float64 3MB 80.0 81.0 ... 80.0 80.0
Attributes: (12/44)
    Conventions:            CF-1.7 CMIP-6.2
    activity_id:            CMIP
    creation_date:          2026-02-11T06:19:45Z
    data_specs_version:     01.00.33
    experiment:             all-forcing simulation of the recent past
    experiment_id:          historical
    ...                     ...
    branch_method:          standard
    external_variables:     areacello
    creator_name:           Romain Beucher
    creator_organisation:   ACCESS-NRI
    creator_email:          romain.beucher@anu.edu.au
    creator_url:            0000-0003-3891-5444

In [30]:
cmoriser.write()

📦 Dataset size: 0.01 GB
   Using chunked writing with DatasetChunker
CMORised output written to deptho_Ofx_ACCESS-ESM1-5_historical_r1i1p1f1_gn_fx.nc
📁 Optimized layout: metadata → data chunks
🗜️ HDF5 compression: shuffle + zlib(level 4) + fletcher32 for data variables


## Test masscello

In [31]:
files = [ROOT_FOLDER + "/output401/ocean/ocean-3d-rho_dzt-1yearly-mean-ym_3077_07.nc"]

In [32]:
cmoriser = ACCESS_ESM_CMORiser(
    input_data=files,
    compound_name="Ofx.masscello",
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
    parent_info=parent_experiment_config,  # <-- This is optional, can be skipped if not needed
)

In [33]:
cmoriser.run()

✓ Skipping frequency validation for time-independent variable
🔧 Applying intelligent dataset rechunking...
🔧 Applying dataset rechunking with rules:
  - Time coordinates: single chunk
  - Time bounds: single chunk
  - Data variables: at least 4.0MB chunks
  rho_dzt: data variable → time:1, st_ocean:50, yt_ocean:300, xt_ocean:360
  xt_ocean: data variable → xt_ocean:360
  yt_ocean: data variable → yt_ocean:300
  st_ocean: data variable → st_ocean:50
  time: coordinate → single chunk
✅ Dataset rechunking completed
✅ Dataset rechunking completed
🔧 Normalizing missing values to NaN for consistent processing...
✅ Missing values normalized to NaN - XArray will handle propagation correctly
🔧 Applying final CMIP6 missing value standardization for masscello...
✅ Final CMIP6 missing value applied: 1e+20


In [34]:
ds = cmoriser.to_dataset()

In [36]:
cmoriser.run()

✓ Skipping frequency validation for time-independent variable
🔧 Applying intelligent dataset rechunking...
🔧 Applying dataset rechunking with rules:
  - Time coordinates: single chunk
  - Time bounds: single chunk
  - Data variables: at least 4.0MB chunks
  rho_dzt: data variable → time:1, st_ocean:50, yt_ocean:300, xt_ocean:360
  xt_ocean: data variable → xt_ocean:360
  yt_ocean: data variable → yt_ocean:300
  st_ocean: data variable → st_ocean:50
  time: coordinate → single chunk
✅ Dataset rechunking completed
✅ Dataset rechunking completed
🔧 Normalizing missing values to NaN for consistent processing...
✅ Missing values normalized to NaN - XArray will handle propagation correctly
🔧 Applying final CMIP6 missing value standardization for masscello...
✅ Final CMIP6 missing value applied: 1e+20


In [37]:
cmoriser.write()

📦 Dataset size: 0.04 GB
   Using chunked writing with DatasetChunker
CMORised output written to masscello_Ofx_ACCESS-ESM1-5_historical_r1i1p1f1_gn_fx.nc
📁 Optimized layout: metadata → data chunks
🗜️ HDF5 compression: shuffle + zlib(level 4) + fletcher32 for data variables
